# 1.1 Neural Network Basics

![The four building blocks of a deep-learning model](assets/pytorch-bubbles.png)

Training a deep learning model requires four building blocks:

1. [**Data**](#Data) — input–target pairs $\{x_i,y_i\}$
2. [**Model**](#building-the-network-using-nnmodule) — a parameterised function $f_\theta$ that maps inputs to predictions, $\hat y_i=f_\theta(x_i)$
3. [**Loss**](#training-the-network) — a function that measures error between predictions and targets
4. [**Optimiser**](#training-the-network) — an algorithm that updates $\theta$ to reduce the loss.

To start, we must import the relevant `torch` and `torchvision` libraries 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
print("PyTorch version:", torch.__version__)

## Data

![data](assets/pytorch-bubbles-data.png)

The cell below loads the famous MNIST dataset using the torchvision library

In [ ]:
# Load the MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# We load a training set
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)

# And withhold some data for testing
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)

In [ ]:
# This visualises random samples from the training dataset
fig, ax = plt.subplots(1, 5, figsize=(15, 3))
for i, idx in enumerate(torch.randint(low=0,high=len(train_dataset),size=(5,))):
    img = train_dataset.data[idx]
    ax[i].imshow(img, cmap='gray')
    ax[i].axis('off')

In [ ]:
# We need to know the shape of the data (i.e. number of pixels) so we can 
# specify the correct number of inputs to our model
img.shape

# Building the Network using `nn.Module` 

![model](assets/pytorch-bubbles-model.png)

To define our own model, we create a class that inherits from `nn.Module`.

The class has two methods that must be defined:

1. **`__init__`** constructs and stores the layers. These layers contain the parameters that will be learned during training. 
<!-- Parameters can be declared during initialisation using `nn.Parameter()`, this tells PyTorch that this is a trainable parameter, registering them with the module and requiring gradients. If other PyTorch modules are declared, their parameters will be registered automatically. -->
2. **`forward`** defines the architecture of the neural network, i.e. how the input $x$ is transformed by the layers and activations to the ouput.

The call to `super().__init__()` is essential - this initialises the nn.Module letting PyTorch know this class is a torch model.
<!-- initialises the machinery that lets PyTorch register submodules and parameters. -->

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

You can now create an instance of this class, and view the architecture by printing the object

In [ ]:
model = SimpleNet()
print(model)

# Training the Network

| **Loss** - `nn.CrossEntropy` | **Optimiser** - `optim.Adam` |
|---|---|
| ![Loss](assets/pytorch-bubbles-loss.png) | ![Optimiser](assets/pytorch-bubbles-opt.png) |

We now want to train the weights of the neural network. For this purpose, we first need to define what a good fit to data entails.

In [ ]:
criterion = nn.CrossEntropyLoss()

Secondly, we need to decide how we want to optimise the network.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.01)

Now we are ready to train the neural network. Let's train for 5 epochs. Note that fixing the number of epochs in this manner is not a good choice and will lead to under- or overfitting. But in this notebook, we merely want to understand the basic concepts behind training a neural network.

In each epoch, for each batch from the training set, we

1) clear the accumulated gradients of the model parameters (accumulated during backpropagation),

2) perform a forward run (i.e. we predict labels for the batch of images),

3) compute the loss for these predictions,

3) perform backpropagation,

5) and use this to optimise the weights and biases of our model.

In [ ]:
N_EPOCHS = 5

model.train()
for epoch in range(N_EPOCHS):
    running_loss = 0.

    for images, labels in train_loader:
        # 1. Zero the gradients
        optimizer.zero_grad()

        # 2. Forward pass
        predictions = model(images)

        # 3. Compute the loss
        loss = criterion(predictions, labels)

        # 4. Compute gradients via backpropagation
        loss.backward()

        # 5. Update parameters
        optimizer.step()

        running_loss += loss.item()

    # Print the average loss for the epoch
    average_loss = running_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{N_EPOCHS}], Loss: {average_loss:.3f}')


## Testing the network

Having trained the network, we can now test it on unseen data (the test set).

In [ ]:
model.eval()  # Set the model to evaluation mode
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        

accuracy = correct / total
print(f'Accuracy on the test set: {100 * accuracy:.1f}%')

That's a pretty high accuracy, reflecting that the MNIST dataset is relatively simple and well-behaved. Indeed, you can do much better. CNNs can reach an accuracy of more than 99 per cent (see [Kaggle](https://www.kaggle.com/code/cdeotte/how-to-choose-cnn-architecture-mnist)). To achieve such a high accuracy, we need to improve the architecture. We will do so in notebook 2.1.

# Exercises

**Exercise 1**: Here, we use the adam optimiser (`optim.Adam`). Find out what other optimisers are available.

**Exercise 2**: What do the hyperparameters (e.g. lr in adam) mean? Change lr to a higher value (e.g. 1) and rerun the notebook. Or to a lower value. What happens? Why?

**Exercise 3**: Here, we use the cross entropy (`nn.CrossEntropyLoss`) as the loss function. What is a loss function? And what is the cross entropy?




**Exercise 4**: Change the architecture of the neural network. Add more layers, increase the number of neurons, or try a different activation function. 